# Lab 5 – Feature Engineering (Classification)
**Dataset:** Medical Insurance Cost (`insurance.csv`)  
**Target:** `smoker` — Binary classification (yes = 1 / no = 0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('insurance.csv')
print(f'Shape: {df.shape}')
df.head()

## Baseline Setup

In [ ]:
def build_features(df, bmi_thresholds=(18.5, 25.0, 30.0), age_bins=4, top_k_region=4):
    d = df.copy()
    # Encode binary categoricals
    d['sex_enc'] = (d['sex'] == 'male').astype(int)
    # BMI category with configurable thresholds
    t1, t2, t3 = bmi_thresholds
    d['bmi_category'] = pd.cut(d['bmi'],
                                bins=[-np.inf, t1, t2, t3, np.inf],
                                labels=['underweight','normal','overweight','obese'])
    d = pd.get_dummies(d, columns=['bmi_category'], drop_first=True)
    # Age group
    d['age_group'] = pd.cut(d['age'], bins=age_bins, labels=False)
    # Region encoding (keep top_k regions, collapse rest to 'other')
    top_regions = d['region'].value_counts().nlargest(top_k_region).index
    d['region_red'] = d['region'].apply(lambda x: x if x in top_regions else 'other')
    d = pd.get_dummies(d, columns=['region_red'], drop_first=True)
    d.drop(columns=['sex','region'], inplace=True)
    return d

def run_model(df_feat, target='smoker', test_size=0.2, seed=42):
    y = (df_feat[target] == 'yes').astype(int)
    X = df_feat.drop(columns=[target, 'charges'])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size,
                                           random_state=seed, stratify=y)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = accuracy_score(yte, m.predict(Xte))
    return acc, m, X.columns.tolist()

df_base = build_features(df)
baseline_acc, baseline_model, baseline_cols = run_model(df_base)
print(f'Baseline Accuracy: {baseline_acc:.4f}')

---
## Task 1 – Create a New Engineered Feature to Predict `smoker`

**New feature: `charges_per_bmi`** = `charges / bmi`

**Justification:** Smokers incur dramatically higher medical charges regardless of BMI, but the relationship between charges and BMI is non-linear — smokers with high BMI are in the most expensive cluster (charges can exceed $40,000). Normalising charges by BMI creates a feature that amplifies this signal: a non-smoker with high BMI will have a moderate ratio, while a smoker with high BMI will have a very large ratio. This gives the classifier a sharper boundary between the two classes compared to using `charges` or `bmi` in isolation, especially at the decision boundary where moderate-charge, moderate-BMI cases are ambiguous.

In [ ]:
df_t1 = build_features(df)
df_t1['charges_per_bmi'] = df_t1['charges'] / (df_t1['bmi'] + 1e-9)

acc_t1, _, _ = run_model(df_t1)
print(f'Baseline:                   {baseline_acc:.4f}')
print(f'With charges_per_bmi:       {acc_t1:.4f}')
print(f'Change:                     {acc_t1 - baseline_acc:+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
smoker_label = (df['smoker'] == 'yes')
axes[0].scatter(df_t1.loc[~smoker_label,'charges'], df_t1.loc[~smoker_label,'bmi'],
                alpha=0.3, s=15, color='steelblue', label='Non-Smoker')
axes[0].scatter(df_t1.loc[smoker_label,'charges'],  df_t1.loc[smoker_label,'bmi'],
                alpha=0.3, s=15, color='tomato', label='Smoker')
axes[0].set_xlabel('charges'); axes[0].set_ylabel('bmi')
axes[0].set_title('charges vs bmi'); axes[0].legend()

axes[1].hist(df_t1.loc[~smoker_label,'charges_per_bmi'], bins=40,
             alpha=0.6, color='steelblue', label='Non-Smoker')
axes[1].hist(df_t1.loc[smoker_label,'charges_per_bmi'],  bins=40,
             alpha=0.6, color='tomato', label='Smoker')
axes[1].set_xlabel('charges_per_bmi')
axes[1].set_title('charges_per_bmi Distribution by Class')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## Task 2 – Change the BMI Category Rule

In [ ]:
# Original rule: underweight < 18.5, normal 18.5-25, overweight 25-30, obese >= 30
# New rule: fine-grained WHO classification
#   underweight < 18.5, normal 18.5-24.9, pre-obese 25-29.9, obese-I 30-34.9, obese-II >= 35

def build_features_v2(df):
    d = df.copy()
    d['sex_enc'] = (d['sex'] == 'male').astype(int)
    d['bmi_cat'] = pd.cut(d['bmi'],
                           bins=[-np.inf, 18.5, 24.9, 29.9, 34.9, np.inf],
                           labels=['underweight','normal','pre_obese','obese_I','obese_II'])
    d = pd.get_dummies(d, columns=['bmi_cat'], drop_first=True)
    d['age_group'] = pd.cut(d['age'], bins=4, labels=False)
    d['region_red'] = d['region']
    d = pd.get_dummies(d, columns=['region_red'], drop_first=True)
    d.drop(columns=['sex','region'], inplace=True)
    return d

df_v2 = build_features_v2(df)
acc_v2, _, _ = run_model(df_v2)

print(f'Original BMI rule (4 bins): {baseline_acc:.4f}')
print(f'Fine-grained rule (5 bins): {acc_v2:.4f}')
print(f'Difference:                 {acc_v2 - baseline_acc:+.4f}')
print()
print('Discussion:')
print('The finer WHO classification adds an extra boundary at BMI=35, separating severely obese')
print('patients. If accuracy improves, the model benefits from this additional distinction.')
print('If unchanged, BMI threshold granularity is not a decisive factor for predicting smoking.')

---
## Task 3 – Vary the Number of Age Groups (top_k analog)

In [ ]:
results = []
for n_bins in [2, 4, 6, 8, 10]:
    d = df.copy()
    d['sex_enc']    = (d['sex']    == 'male').astype(int)
    d['bmi_category'] = pd.cut(d['bmi'],bins=[-np.inf,18.5,25,30,np.inf],
                                labels=['underweight','normal','overweight','obese'])
    d = pd.get_dummies(d, columns=['bmi_category'], drop_first=True)
    d['age_group']  = pd.cut(d['age'], bins=n_bins, labels=False)
    d = pd.get_dummies(d, columns=['region'], drop_first=True)
    d.drop(columns=['sex'], inplace=True)
    acc, m, cols = run_model(d)
    top3 = pd.Series(m.feature_importances_, index=m.feature_names_in_).nlargest(3).index.tolist()
    results.append({'age_bins': n_bins, 'accuracy': round(acc,4), 'top_3_features': top3})

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.plot(res_df['age_bins'], res_df['accuracy'], marker='o', color='steelblue')
plt.xlabel('Number of Age Bins'); plt.ylabel('Accuracy')
plt.title('Accuracy vs Number of Age Groups')
plt.xticks(res_df['age_bins'])
plt.tight_layout(); plt.show()

---
## Task 4 – Feature Selection (Optional)

In [ ]:
df_fs = build_features(df)
df_fs['charges_per_bmi'] = df_fs['charges'] / (df_fs['bmi'] + 1e-9)
y_fs  = (df_fs['smoker'] == 'yes').astype(int)
X_fs  = df_fs.drop(columns=['smoker','charges'])

Xtr, Xte, ytr, yte = train_test_split(X_fs, y_fs, test_size=0.2, random_state=42, stratify=y_fs)

rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(Xtr, ytr)
acc_full = accuracy_score(yte, rf_full.predict(Xte))

selector  = SelectFromModel(rf_full, prefit=True)
selected  = X_fs.columns[selector.get_support()].tolist()
rf_sel    = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sel.fit(selector.transform(Xtr), ytr)
acc_sel   = accuracy_score(yte, rf_sel.predict(selector.transform(Xte)))

print(f'All features ({X_fs.shape[1]}):     accuracy = {acc_full:.4f}')
print(f'Selected features ({len(selected)}): accuracy = {acc_sel:.4f}')
print(f'Selected: {selected}')
print()

importances = pd.Series(rf_full.feature_importances_, index=X_fs.columns).sort_values(ascending=True)
importances.tail(10).plot(kind='barh', figsize=(8,5), color='steelblue', edgecolor='black')
plt.title('Top Feature Importances')
plt.tight_layout(); plt.show()

**Feature Selection Analysis:**  
`charges` and `charges_per_bmi` dominate the importance scores — this makes intuitive sense because smokers are in a completely different price bracket. `bmi` and `age` contribute moderately. One-hot encoded region and sex features contribute very little. Feature selection reduces the model to the most informative features without sacrificing accuracy, confirming that for this specific target (`smoker`), a small subset of financial/health features is sufficient.